In [ ]:
from sklearn.datasets import fetch_20newsgroups

dataset = fetch_20newsgroups(
    subset="all",
    remove=("headers", "footers", "quotes")
)

documents = dataset.data
labels = dataset.target

print(len(documents))

In [ ]:
!pip install -q sentence-transformers faiss-cpu scikit-learn scikit-fuzzy fastapi uvicorn nest-asyncio

In [ ]:
import numpy as np
import pandas as pd
import re
import string

from sklearn.datasets import fetch_20newsgroups
from sentence_transformers import SentenceTransformer

import faiss
import skfuzzy as fuzz

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
print(documents[0])

In [ ]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r'\d+', '', text)

    text = text.translate(str.maketrans('', '', string.punctuation))

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
clean_docs = [clean_text(doc) for doc in documents]

print(clean_docs[0])

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
embeddings = model.encode(
    clean_docs,
    batch_size=64,
    show_progress_bar=True
)

In [ ]:
print(embeddings.shape)

In [ ]:
np.save("embeddings.npy", embeddings)

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings).astype("float32"))

print("Total vectors in database:", index.ntotal)

In [ ]:
def search_documents(query, k=5):

    query_embedding = model.encode([query])

    D, I = index.search(
        np.array(query_embedding).astype("float32"), k
    )

    results = [documents[i] for i in I[0]]

    return results

In [ ]:
search_documents("space rocket launch")

In [ ]:
data = embeddings.T

In [ ]:
n_clusters = 15

cntr, u, _, _, _, _, _ = fuzz.cluster.cmeans(
    data,
    n_clusters,
    2,
    error=0.005,
    maxiter=1000
)

In [ ]:
print(u.shape)

In [ ]:
def dominant_cluster(doc_id):

    return np.argmax(u[:, doc_id])

In [ ]:
def cosine_similarity(a, b):

    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
class SemanticCache:

    def __init__(self, threshold=0.85):

        self.cache = []
        self.threshold = threshold

        self.hit_count = 0
        self.miss_count = 0


    def lookup(self, query_embedding):

        for entry in self.cache:

            similarity = cosine_similarity(query_embedding, entry["embedding"])

            if similarity > self.threshold:

                self.hit_count += 1

                return True, entry, similarity

        self.miss_count += 1

        return False, None, 0


    def add(self, query, embedding, result):

        self.cache.append({
            "query": query,
            "embedding": embedding,
            "result": result
        })


    def stats(self):

        total = self.hit_count + self.miss_count

        hit_rate = self.hit_count / total if total > 0 else 0

        return {
            "total_entries": len(self.cache),
            "hit_count": self.hit_count,
            "miss_count": self.miss_count,
            "hit_rate": hit_rate
        }


    def clear(self):

        self.cache = []

        self.hit_count = 0
        self.miss_count = 0

In [ ]:
cache = SemanticCache(threshold=0.85)

In [ ]:
def query_system(query):

    query_embedding = model.encode(query)

    hit, entry, similarity = cache.lookup(query_embedding)

    if hit:

        return {
            "query": query,
            "cache_hit": True,
            "matched_query": entry["query"],
            "similarity_score": similarity,
            "result": entry["result"]
        }

    results = search_documents(query)

    cache.add(query, query_embedding, results)

    return {
        "query": query,
        "cache_hit": False,
        "result": results
    }

In [ ]:
query_system("space exploration")

In [ ]:
query_system("exploring space missions")

In [ ]:
cache.stats()

In [ ]:
cache.clear()

In [ ]:
!pip install fastapi uvicorn nest-asyncio pyngrok

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import nest_asyncio
import uvicorn

nest_asyncio.apply()

In [ ]:
app = FastAPI()

In [ ]:
class QueryRequest(BaseModel):
    query: str

In [ ]:
@app.post("/query")
def search_api(request: QueryRequest):

    query = request.query

    query_embedding = model.encode(query)

    hit, entry, similarity = cache.lookup(query_embedding)

    if hit:

        return {
            "query": query,
            "cache_hit": True,
            "matched_query": entry["query"],
            "similarity_score": float(similarity),
            "result": entry["result"]
        }

    results = search_documents(query)

    cache.add(query, query_embedding, results)

    return {
        "query": query,
        "cache_hit": False,
        "result": results
    }

In [ ]:
@app.get("/cache/stats")
def cache_stats():

    return cache.stats()

In [ ]:
@app.delete("/cache")
def clear_cache():

    cache.clear()

    return {"message": "Cache cleared"}

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
import uvicorn
import threading

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_api)
thread.start()

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3AdSJG7e86gd2fFii9A2JEOc3X4_TN4Me5zARfTbfQWPD8B")

public_url = ngrok.connect(8000)

print("Public API URL:", public_url)